In [ ]:
# One-time install
# pip install vllm langchain-openai
#
# Start vLLM server in terminal before running this notebook:
# vllm serve Qwen/Qwen3-8B \
#   --enable-auto-tool-choice \
#   --tool-call-parser hermes \
#   --gpu-memory-utilization 0.90 \
#   --max-model-len 8192 \
#   --port 8000

In [ ]:
from dotenv import load_dotenv
from dataclasses import dataclass
from langchain_community.utilities import SQLDatabase
from langchain_core.tools import tool
from langgraph.runtime import get_runtime
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI  # <-- replaces ChatOllama

load_dotenv()

In [ ]:
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [ ]:
@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [ ]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- Call execute_sql as many times as needed to answer the question.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 50 rows of output unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
- Answer concisely — give only what was asked. Do not add rankings or
  extra data unless the user explicitly requests them.
"""

In [ ]:
# ChatOpenAI pointed at local vLLM server
# No reasoning hacks, no num_ctx workarounds — vLLM handles everything
agent = create_agent(
    model=ChatOpenAI(
        model="Qwen/Qwen2.5-3B-Instruct-AWQ",
        base_url="http://localhost:8000/v1",
        api_key="not-needed",
        temperature=0,
    ),
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

In [ ]:
question = "Which table has the largest number of entries?"
for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()